# PPO (Proximal Policy Optimization)

本notebook演示PPO算法的实现和应用。

PPO是RLHF中最常用的强化学习算法，通过Clip机制保证训练稳定性。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional

np.random.seed(42)

## 1. PPO训练器实现

In [ ]:
class PPOTrainer:
    """PPO训练器"""
    
    def __init__(
        self,
        clip_epsilon: float = 0.2,
        value_loss_coef: float = 0.5,
        entropy_coef: float = 0.01,
        gamma: float = 1.0,
        lam: float = 0.95
    ):
        self.clip_epsilon = clip_epsilon
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.gamma = gamma
        self.lam = lam
    
    def compute_advantages(
        self,
        rewards: np.ndarray,
        values: np.ndarray,
        dones: Optional[np.ndarray] = None
    ) -> Tuple[np.ndarray, np.ndarray]:
        """使用GAE计算优势函数"""
        T = len(rewards)
        advantages = np.zeros(T)
        returns = np.zeros(T)
        
        if dones is None:
            dones = np.zeros(T)
        
        gae = 0
        for t in reversed(range(T)):
            if t == T - 1:
                next_value = 0 if dones[t] else values[t + 1]
            else:
                next_value = values[t + 1] * (1 - dones[t])
            
            delta = rewards[t] + self.gamma * next_value - values[t]
            gae = delta + self.gamma * self.lam * (1 - dones[t]) * gae
            advantages[t] = gae
            returns[t] = advantages[t] + values[t]
        
        return advantages, returns
    
    def ppo_loss(
        self,
        old_log_probs: np.ndarray,
        new_log_probs: np.ndarray,
        advantages: np.ndarray,
        old_values: np.ndarray,
        new_values: np.ndarray,
        returns: np.ndarray,
        entropy: Optional[np.ndarray] = None
    ) -> Tuple[float, Dict[str, float]]:
        """计算PPO损失"""
        # 标准化优势函数
        advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        
        # 1. 策略损失（PPO-Clip）
        ratio = np.exp(new_log_probs - old_log_probs)
        surr1 = ratio * advantages
        surr2 = np.clip(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages
        policy_loss = -np.mean(np.minimum(surr1, surr2))
        
        # 2. 价值函数损失
        value_pred_clipped = old_values + np.clip(
            new_values - old_values,
            -self.clip_epsilon,
            self.clip_epsilon
        )
        value_loss_unclipped = (new_values - returns) ** 2
        value_loss_clipped = (value_pred_clipped - returns) ** 2
        value_loss = 0.5 * np.mean(np.maximum(value_loss_unclipped, value_loss_clipped))
        
        # 3. 熵损失
        if entropy is None:
            entropy = np.zeros_like(advantages)
        entropy_loss = -np.mean(entropy)
        
        # 总损失
        total_loss = (
            policy_loss +
            self.value_loss_coef * value_loss +
            self.entropy_coef * entropy_loss
        )
        
        # KL散度
        kl = np.mean(old_log_probs - new_log_probs)
        
        metrics = {
            'loss/total': total_loss,
            'loss/policy': policy_loss,
            'loss/value': value_loss,
            'loss/entropy': entropy_loss,
            'policy/approx_kl': kl,
            'policy/clip_fraction': np.mean(np.abs(ratio - 1.0) > self.clip_epsilon),
            'policy/ratio_mean': np.mean(ratio),
        }
        
        return total_loss, metrics

## 2. 创建PPO训练器

In [ ]:
ppo = PPOTrainer(
    clip_epsilon=0.2,
    value_loss_coef=0.5,
    entropy_coef=0.01,
    gamma=1.0,
    lam=0.95
)

print("PPO配置:")
print(f"  clip_epsilon: {ppo.clip_epsilon}")
print(f"  value_loss_coef: {ppo.value_loss_coef}")
print(f"  entropy_coef: {ppo.entropy_coef}")
print(f"  gamma: {ppo.gamma}")
print(f"  lam: {ppo.lam}")

## 3. GAE优势函数计算

In [ ]:
# 模拟一个trajectory
T = 10
rewards = np.random.randn(T) * 0.5 + 0.5
values = np.random.randn(T + 1) * 0.3

# 计算优势函数
advantages, returns = ppo.compute_advantages(rewards, values)

print(f"Trajectory长度: {T}")
print(f"\n奖励: {rewards}")
print(f"\n优势函数: {advantages}")
print(f"\n回报: {returns}")
print(f"\n优势均值: {np.mean(advantages):.4f}")
print(f"优势标准差: {np.std(advantages):.4f}")

## 4. PPO损失计算

In [ ]:
batch_size = 32

# 模拟数据
old_log_probs = np.random.randn(batch_size) * 0.5 - 2.0
new_log_probs = old_log_probs + np.random.randn(batch_size) * 0.1
advantages_batch = np.random.randn(batch_size)
old_values = np.random.randn(batch_size)
new_values = old_values + np.random.randn(batch_size) * 0.1
returns_batch = old_values + advantages_batch
entropy_batch = -new_log_probs

# 计算损失
loss, metrics = ppo.ppo_loss(
    old_log_probs,
    new_log_probs,
    advantages_batch,
    old_values,
    new_values,
    returns_batch,
    entropy_batch
)

print(f"总损失: {loss:.4f}")
print(f"\n详细指标:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

## 5. PPO Clip机制可视化

In [ ]:
# 生成不同的ratio值
ratios = np.linspace(0.5, 1.5, 200)
clip_epsilon = 0.2

# 正优势
advantage_pos = 1.0
surr1_pos = ratios * advantage_pos
surr2_pos = np.clip(ratios, 1 - clip_epsilon, 1 + clip_epsilon) * advantage_pos
objective_pos = np.minimum(surr1_pos, surr2_pos)

# 负优势
advantage_neg = -1.0
surr1_neg = ratios * advantage_neg
surr2_neg = np.clip(ratios, 1 - clip_epsilon, 1 + clip_epsilon) * advantage_neg
objective_neg = np.minimum(surr1_neg, surr2_neg)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 正优势
axes[0].plot(ratios, surr1_pos, 'b--', label='未clip (ratio × A)', alpha=0.6)
axes[0].plot(ratios, surr2_pos, 'r--', label='clip后', alpha=0.6)
axes[0].plot(ratios, objective_pos, 'g-', linewidth=2, label='PPO目标 (min)')
axes[0].axvline(x=1-clip_epsilon, color='gray', linestyle=':', alpha=0.5)
axes[0].axvline(x=1+clip_epsilon, color='gray', linestyle=':', alpha=0.5)
axes[0].axvline(x=1.0, color='black', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Ratio (π_new / π_old)')
axes[0].set_ylabel('目标函数值')
axes[0].set_title('正优势 (A > 0)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 负优势
axes[1].plot(ratios, surr1_neg, 'b--', label='未clip (ratio × A)', alpha=0.6)
axes[1].plot(ratios, surr2_neg, 'r--', label='clip后', alpha=0.6)
axes[1].plot(ratios, objective_neg, 'g-', linewidth=2, label='PPO目标 (min)')
axes[1].axvline(x=1-clip_epsilon, color='gray', linestyle=':', alpha=0.5)
axes[1].axvline(x=1+clip_epsilon, color='gray', linestyle=':', alpha=0.5)
axes[1].axvline(x=1.0, color='black', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Ratio (π_new / π_old)')
axes[1].set_ylabel('目标函数值')
axes[1].set_title('负优势 (A < 0)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Clip机制解释:")
print("  • 正优势时: ratio > 1+ε 被clip，防止过度增强")
print("  • 负优势时: ratio < 1-ε 被clip，防止过度惩罚")
print("  • 保证策略更新不会太激进")

## 6. 不同clip_epsilon的影响

In [ ]:
epsilons = [0.1, 0.2, 0.3, 0.5]
losses = []
clip_fractions = []
kls = []

for eps in epsilons:
    ppo_temp = PPOTrainer(clip_epsilon=eps)
    loss_temp, metrics_temp = ppo_temp.ppo_loss(
        old_log_probs,
        new_log_probs,
        advantages_batch,
        old_values,
        new_values,
        returns_batch,
        entropy_batch
    )
    losses.append(loss_temp)
    clip_fractions.append(metrics_temp['policy/clip_fraction'])
    kls.append(metrics_temp['policy/approx_kl'])

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epsilons, losses, 'o-')
axes[0].set_xlabel('Clip Epsilon')
axes[0].set_ylabel('Loss')
axes[0].set_title('损失 vs Epsilon')
axes[0].grid(True)

axes[1].plot(epsilons, clip_fractions, 'o-', color='green')
axes[1].set_xlabel('Clip Epsilon')
axes[1].set_ylabel('Clip Fraction')
axes[1].set_title('Clip比例 vs Epsilon')
axes[1].grid(True)

axes[2].plot(epsilons, kls, 'o-', color='red')
axes[2].set_xlabel('Clip Epsilon')
axes[2].set_ylabel('KL Divergence')
axes[2].set_title('KL散度 vs Epsilon')
axes[2].grid(True)

plt.tight_layout()
plt.show()

print("观察:")
print("  • ε越小: 更保守，clip比例越高")
print("  • ε越大: 更激进，允许更大更新")
print("  • 推荐值: ε = 0.2（论文推荐）")

## 7. RLHF中的PPO流程

In [ ]:
print("RLHF训练流程:\n")
print("第1步: SFT (Supervised Fine-Tuning)")
print("  - 在指令数据上微调基础模型")
print("  - 得到初始策略π_0\n")

print("第2步: RM (Reward Model Training)")
print("  - 收集人类偏好数据")
print("  - 训练奖励模型r(x, y)\n")

print("第3步: PPO (强化学习优化)")
print("  For each iteration:")
print("    1. 采样: 从当前策略π生成响应")
print("    2. 评分: 使用奖励模型r打分")
print("    3. 计算: GAE优势函数")
print("    4. 优化: PPO更新策略")
print("    5. 约束: KL散度限制偏离π_ref\n")

print("在LLM中的映射:")
print("  • 状态(s): 输入提示")
print("  • 动作(a): 生成的token")
print("  • 策略(π): 语言模型")
print("  • 奖励(r): 奖励模型评分 + KL惩罚")
print("  • 价值函数(V): 预测累积奖励")

## 8. PPO vs 其他算法对比

In [ ]:
import pandas as pd

comparison_data = {
    '算法': ['PPO', 'TRPO', 'A3C', 'DQN', 'DPO'],
    '稳定性': ['高', '很高', '中', '中', '很高'],
    '样本效率': ['中-高', '中', '中', '低', '高'],
    '实现难度': ['中', '复杂', '中', '简单', '简单'],
    'RLHF适用': ['是', '是', '否', '否', '是']
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

print("\nPPO在RLHF中的优势:")
print("  ✓ 训练稳定（Clip机制）")
print("  ✓ 样本效率高（可重用数据）")
print("  ✓ 超参数鲁棒")
print("  ✓ 广泛验证（ChatGPT等）")

## 9. 实际应用案例

In [ ]:
print("PPO在LLM对齐中的成功案例:\n")

cases = [
    {
        'name': 'ChatGPT',
        'model': 'GPT-3.5',
        'method': 'SFT + RM + PPO',
        'result': '达到人类水平的对话能力'
    },
    {
        'name': 'GPT-4',
        'model': 'GPT-4',
        'method': 'RLHF (PPO)',
        'result': '超越人类平均水平'
    },
    {
        'name': 'Claude',
        'model': 'Claude 2/3',
        'method': 'Constitutional AI + PPO',
        'result': '安全、有帮助的对话'
    },
    {
        'name': 'Llama 2-Chat',
        'model': 'Llama 2',
        'method': 'RLHF (PPO)',
        'result': '开源对话模型标杆'
    }
]

for i, case in enumerate(cases, 1):
    print(f"{i}. {case['name']}")
    print(f"   基座: {case['model']}")
    print(f"   方法: {case['method']}")
    print(f"   结果: {case['result']}\n")

print("典型PPO配置（LLM对齐）:")
config = {
    'clip_epsilon': 0.2,
    'value_loss_coef': 0.5,
    'entropy_coef': 0.01,
    'kl_coef': 0.1,
    'learning_rate': 1e-6,
    'batch_size': '64-512',
    'ppo_epochs': 4,
    'sequence_length': '512-2048'
}

for key, value in config.items():
    print(f"  • {key}: {value}")

## 10. 总结

In [ ]:
print("PPO的关键要点:\n")

print("核心机制:")
print("  • Clip目标函数: 限制策略更新幅度")
print("  • GAE: 高效计算优势函数")
print("  • 价值函数: 估计累积奖励")
print("  • 熵正则化: 鼓励探索\n")

print("优势:")
print("  ✓ 训练稳定（Clip机制）")
print("  ✓ 样本效率高（可重用数据）")
print("  ✓ 超参数鲁棒")
print("  ✓ RLHF标准算法\n")

print("局限性:")
print("  ✗ 比DPO复杂（需要RM + RL）")
print("  ✗ 训练时间较长")
print("  ✗ 可能出现reward hacking\n")

print("最佳实践:")
print("  • 从SFT模型初始化")
print("  • 使用较小学习率")
print("  • 监控KL散度")
print("  • 使用梯度裁剪")
print("  • 标准化优势函数")